# Create LaTeX Decoder Ablation Tables

Generate booktabs-style LaTeX tables of the segmentation decoder ablation: the
overall pathological and anatomical segmentation scores of each model trained
with the SegFormer decoder and with the upsampler decoder. Each cell reports the
mean positive Dice over the tasks of a category and the standard error of the
evaluation protocol, which combines the variation over the training runs with
the sampling noise of the test cohorts.

Two tables are produced, one for full fine-tuning and one for head-only
training. The generated tables require `\usepackage{booktabs}`,
`\usepackage{siunitx}` and `\usepackage{graphicx}` in the document preamble.

In [ ]:
from pathlib import Path

import pandas as pd

# Only the models trained with both decoders are tabulated, in this order.
MODEL_DISPLAY_NAMES = {
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
}

CATEGORY_DISPLAY_NAMES = {
    "pathology_segmentation": "Pathological",
    "anatomy_segmentation": "Anatomical",
}

DECODER_DISPLAY_NAMES = {
    "segformer": "SegFormer",
    "upsampler": "Upsampler",
}

PARADIGM_DISPLAY_NAMES = {
    "full_finetuning": "full fine-tuning",
    "head_only": "head-only training",
}

# The category scores of both decoders side by side, exported by
# `simplified_protocol.ipynb`.
COMPARISON_PATH = Path("simplified_protocol_decoder_comparison.csv")

# The siunitx setup for the metric columns. `mode = text` and
# `reset-text-series = false` are what let the font change on the best value in
# a column reach the digits; `detect-weight = true` is the siunitx v2 spelling
# of the same thing. `retain-zero-uncertainty` keeps the "+/- 0.000" on the
# cells whose seeds agree exactly, which siunitx otherwise discards.
SISETUP_OPTIONS = [
    "table-format = 1.3(3)",
    "uncertainty-mode = separate",
    "retain-zero-uncertainty = true",
    "mode = text",
    "reset-text-series = false",
]

# `\textbf` selects Computer Modern's bold *extended* face, whose digits are
# 15% wider than the regular face, so an emboldened cell is wider than the rest
# of its column and siunitx cannot align it. Series "b" is bold but not
# extended, and has exactly the regular digit widths, as long as both come from
# the 10pt design size -- hence setting the size explicitly rather than with
# `\footnotesize`. The tables are scaled to the text width afterwards, so this
# is not the size the reader sees.
BOLD_SERIES = r"\fontseries{b}\selectfont"
TABLE_FONT_SIZE = r"\fontsize{10}{12}\selectfont"

## Load the Category Scores

Load the side-by-side category scores exported by `simplified_protocol.ipynb`
and reshape them to one row per category, model, paradigm and decoder. The
standard error is the protocol's category-level one, formed on the category
score so that tasks sharing a test set are not treated as independent.

In [ ]:
def load_category_scores(path: Path = COMPARISON_PATH) -> pd.DataFrame:
    """Load the category scores of both decoders in long form.

    Args:
        path: The decoder comparison exported by `simplified_protocol.ipynb`.

    Returns:
        A DataFrame with one row per category, model, paradigm and decoder, and
        `mean` and `stderr` columns.

    Raises:
        FileNotFoundError: If the comparison has not been exported yet.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run simplified_protocol.ipynb to export it."
        )

    comparison = pd.read_csv(path)
    keys = ["category", "model", "paradigm"]

    return pd.concat(
        [
            comparison[[*keys, f"mean_{decoder}", f"se_{decoder}"]]
            .rename(columns={f"mean_{decoder}": "mean", f"se_{decoder}": "stderr"})
            .assign(decoder=decoder)
            for decoder in DECODER_DISPLAY_NAMES
        ],
        ignore_index=True,
    )


category_scores = load_category_scores()
category_scores.head()

## Build the LaTeX Tables

Each table lists the models as rows and the decoders as columns, grouped by
category. For each model and category, the better of the two decoders is shown
in bold.

In [ ]:
def format_cell(mean: float, stderr: float, is_best: bool) -> str:
    """Format a mean and standard error as a siunitx table cell.

    Args:
        mean: The mean value of the metric.
        stderr: The standard error of the mean value.
        is_best: Whether this is the better decoder for the model and category.

    Returns:
        The cell contents in siunitx uncertainty notation, prefixed with the
        bold font series if it is the better decoder. The prefix is
        deliberately not braced, which would stop siunitx aligning the cell.
    """
    cell = f"{mean:.3f}({stderr:.3f})"

    return f"{BOLD_SERIES} {cell}" if is_best else cell


def build_table_frame(scores: pd.DataFrame, paradigm: str) -> pd.DataFrame:
    """Pivot the category scores into a table of formatted LaTeX cells.

    Args:
        scores: The category scores, as returned by `load_category_scores`.
        paradigm: The training paradigm to tabulate, e.g. `"full_finetuning"`.

    Returns:
        A DataFrame of LaTeX cell strings indexed by model display name, with a
        two-level column index of (category, decoder) display names.

    Raises:
        ValueError: If any model, category or decoder has no result for the
            paradigm.
    """
    subset = scores[scores["paradigm"] == paradigm]
    models = list(MODEL_DISPLAY_NAMES)
    columns = pd.MultiIndex.from_product(
        [list(CATEGORY_DISPLAY_NAMES), list(DECODER_DISPLAY_NAMES)]
    )

    means = subset.pivot(
        index="model", columns=["category", "decoder"], values="mean"
    ).reindex(index=models, columns=columns)
    stderrs = subset.pivot(
        index="model", columns=["category", "decoder"], values="stderr"
    ).reindex(index=models, columns=columns)

    if means.isna().any().any() or stderrs.isna().any().any():
        raise ValueError(f"Missing '{paradigm}' results for some model or decoder.")

    best_decoders = {
        category: means[category].idxmax(axis=1) for category in CATEGORY_DISPLAY_NAMES
    }

    return pd.DataFrame(
        [
            [
                format_cell(
                    means.at[model, (category, decoder)],
                    stderrs.at[model, (category, decoder)],
                    best_decoders[category][model] == decoder,
                )
                for category, decoder in columns
            ]
            for model in models
        ],
        index=[MODEL_DISPLAY_NAMES[model] for model in models],
        columns=pd.MultiIndex.from_tuples(
            [
                (CATEGORY_DISPLAY_NAMES[category], DECODER_DISPLAY_NAMES[decoder])
                for category, decoder in columns
            ]
        ),
    )


def insert_cmidrules(tabular: str, columns: pd.Index) -> str:
    r"""Insert booktabs rules underneath the column group headers.

    Pandas emits `\multicolumn` headers for grouped columns but no accompanying
    `\cmidrule`s, so they are added here.

    Args:
        tabular: The LaTeX tabular environment produced by `Styler.to_latex`.
        columns: The two-level column index of the tabulated DataFrame.

    Returns:
        The tabular environment with a rule under each column group header.

    Raises:
        ValueError: If the tabular environment has no column group header row.
    """
    categories = columns.get_level_values(0)
    rules = []
    column = 2  # The first column holds the model names.
    for category in dict.fromkeys(categories):
        width = sum(category == other for other in categories)
        rules.append(rf"\cmidrule(lr){{{column}-{column + width - 1}}}")
        column += width

    lines = tabular.splitlines()
    header_lines = [i for i, line in enumerate(lines) if r"\multicolumn" in line]
    if not header_lines:
        raise ValueError("No column group header found in the tabular environment.")
    lines.insert(header_lines[0] + 1, " ".join(rules))

    return "\n".join(lines)


def to_booktabs_latex(frame: pd.DataFrame, caption: str, label: str) -> str:
    """Render a table of LaTeX cells as a booktabs-style table environment.

    Args:
        frame: A DataFrame of cell strings from `build_table_frame`.
        caption: The caption of the table.
        label: The label used to reference the table.

    Returns:
        The complete LaTeX source of the table.
    """
    tabular = frame.style.hide(axis="index", names=True).to_latex(
        hrules=True,
        siunitx=True,
        multicol_align="c",
        column_format="l" + "S" * frame.shape[1],
    )
    sisetup = "\n".join(
        [r"\sisetup{", *(f"  {option}," for option in SISETUP_OPTIONS), "}"]
    )

    return "\n".join(
        [
            r"\begin{table}[t]",
            r"\centering",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
            TABLE_FONT_SIZE,
            r"\setlength{\tabcolsep}{3pt}",
            sisetup,
            r"\resizebox{\columnwidth}{!}{%",
            insert_cmidrules(tabular.strip(), frame.columns),
            r"}",
            r"\end{table}",
            "",
        ]
    )

## Generate the Tables

Write each table to a `.tex` file alongside this notebook and print its source.

In [ ]:
for paradigm, paradigm_name in PARADIGM_DISPLAY_NAMES.items():
    frame = build_table_frame(category_scores, paradigm)
    latex = to_booktabs_latex(
        frame,
        caption=(
            "The mean and standard error of the positive Dice score over the "
            "pathological and anatomical segmentation tasks for each model after "
            f"{paradigm_name} with the SegFormer and upsampler decoders. The "
            "better decoder for each model and category is shown in bold."
        ),
        label=f"tab:segmentation_decoder_ablation_{paradigm}",
    )
    Path(f"segmentation_decoder_ablation_{paradigm}_results.tex").write_text(latex)
    print(latex)

## Generate the Markdown Tables

The same tables in Markdown, for the OpenReview rebuttal. Markdown has no
grouped column headers, so each column is named after its category and
decoder, and the standard error follows the mean as `± stderr`. The tables
are written to a `.md` file alongside this notebook, printed for copying, and
rendered below as a preview.

In [ ]:
def format_markdown_cell(mean: float, stderr: float, is_best: bool) -> str:
    """Format a mean and standard error as a Markdown table cell.

    Args:
        mean: The mean value of the metric.
        stderr: The standard error of the mean value.
        is_best: Whether this is the better decoder for the model and category.

    Returns:
        The cell contents as `mean ± stderr`, in bold if it is the better
        decoder.
    """
    cell = f"{mean:.3f} ± {stderr:.3f}"

    return f"**{cell}**" if is_best else cell


def to_markdown_table(scores: pd.DataFrame, paradigm: str) -> str:
    """Render the category scores of a paradigm as a Markdown table.

    Args:
        scores: The category scores, as returned by `load_category_scores`.
        paradigm: The training paradigm to tabulate, e.g. `"full_finetuning"`.

    Returns:
        The Markdown source of the table, with the models as rows and one
        column per category and decoder.

    Raises:
        ValueError: If any model, category or decoder has no result for the
            paradigm.
    """
    subset = scores[scores["paradigm"] == paradigm].set_index(
        ["model", "category", "decoder"]
    )
    expected = pd.MultiIndex.from_product(
        [
            list(MODEL_DISPLAY_NAMES),
            list(CATEGORY_DISPLAY_NAMES),
            list(DECODER_DISPLAY_NAMES),
        ]
    )
    missing = expected.difference(subset.index)
    if len(missing) > 0:
        raise ValueError(f"Missing '{paradigm}' results for {list(missing)}.")

    header = ["Model"] + [
        f"{CATEGORY_DISPLAY_NAMES[category]} ({DECODER_DISPLAY_NAMES[decoder]})"
        for category in CATEGORY_DISPLAY_NAMES
        for decoder in DECODER_DISPLAY_NAMES
    ]
    lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join([":--"] + [":-:"] * (len(header) - 1)) + " |",
    ]

    for model, model_name in MODEL_DISPLAY_NAMES.items():
        cells = [model_name]
        for category in CATEGORY_DISPLAY_NAMES:
            means = {
                decoder: subset.at[(model, category, decoder), "mean"]
                for decoder in DECODER_DISPLAY_NAMES
            }
            best_decoder = max(means, key=means.get)
            cells += [
                format_markdown_cell(
                    means[decoder],
                    subset.at[(model, category, decoder), "stderr"],
                    decoder == best_decoder,
                )
                for decoder in DECODER_DISPLAY_NAMES
            ]
        lines.append("| " + " | ".join(cells) + " |")

    return "\n".join(lines)

In [ ]:
from IPython.display import Markdown, display

sections = []
for paradigm, paradigm_name in PARADIGM_DISPLAY_NAMES.items():
    caption = (
        f"**{paradigm_name.capitalize()}.** The mean ± standard error of the "
        "positive Dice score over the pathological and anatomical segmentation "
        "tasks for each model with the SegFormer and upsampler decoders. The "
        "better decoder for each model and category is shown in bold."
    )
    sections.append(f"{caption}\n\n{to_markdown_table(category_scores, paradigm)}")

markdown_tables = "\n\n".join(sections) + "\n"
Path("segmentation_decoder_ablation_results.md").write_text(markdown_tables)
print(markdown_tables)
display(Markdown(markdown_tables))